# MAFAULDA — STFT 4 cửa sổ + chia Train/Validation/Test + Resume

Notebook này:
- dùng 4 lớp `normal`, `inner_race`, `outer_race`, `ball_fault`;
- dùng `Radial_Acc_Under` (cột 3);
- chia mỗi CSV thành segment 4096 mẫu;
- tạo 4 STFT vuông: 65×65, 129×129, 257×257, 513×513;
- không dùng `np.fft`;
- lưu split cố định bằng `data_split.json`;
- khi chạy lại, kiểm tra **số lượng file NPY** của cả 4 cửa sổ trước khi đọc CSV;
- nếu đủ 61 file/cửa sổ (244 NPY/file tốc độ) thì bỏ qua toàn bộ;
- nếu thiếu thì chỉ tạo phần còn thiếu.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# CẤU HÌNH
# ============================================================

project_folder = Path("/content/drive/MyDrive")

SOURCE_CLASS_DIRS = {
    "normal": project_folder / "normal",
    "inner_race": project_folder / "inner_race",
    "outer_race": project_folder / "outer_race",
    "ball_fault": project_folder / "ball_fault",
}

# GIỮ NGUYÊN đường dẫn này nếu muốn tiếp tục dữ liệu đã tạo trước đó.
OUTPUT_ROOT = project_folder / "stft_dataset_split_balanced"

FS = 50000
SEGMENT_LENGTH = 4096
EXPECTED_SAMPLES_PER_FILE = 250000
EXPECTED_SEGMENTS = EXPECTED_SAMPLES_PER_FILE // SEGMENT_LENGTH

VIBRATION_COLUMN_INDEX = 2

WINDOW_LENGTHS = [128, 256, 512, 1024]

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

SAVE_NPY = True
SAVE_PNG = False
SKIP_EXISTING_NPY = True

MAX_FILES_PER_CLASS = None
MAX_SEGMENTS_PER_FILE = None

COLUMNS = [
    "Tachometer",
    "Axial_Acc_Under",
    "Radial_Acc_Under",
    "Tangential_Acc_Under",
    "Axial_Acc_Over",
    "Radial_Acc_Over",
    "Tangential_Acc_Over",
    "Microphone",
]

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXPECTED_SEGMENTS:", EXPECTED_SEGMENTS)
print("Kênh:", COLUMNS[VIBRATION_COLUMN_INDEX])

In [ ]:
def remove_dc(x):
    mean_value = float(np.mean(x))
    return x - mean_value, mean_value


def create_hamming(Nw):
    w = np.zeros(Nw, dtype=np.float64)

    for n in range(Nw):
        w[n] = 0.54 - 0.46 * np.cos(
            2.0 * np.pi * n / (Nw - 1)
        )

    return w


def calculate_square_parameters(Nx, Nw):
    NFFT = Nw
    F = Nw // 2 + 1

    numerator = Nx - Nw
    denominator = F - 1

    if numerator % denominator != 0:
        raise ValueError(
            f"Nx={Nx}, Nw={Nw}: không tìm được H nguyên."
        )

    H = numerator // denominator
    No = Nw - H
    T = (Nx - Nw) // H + 1

    if T != F:
        raise RuntimeError(
            f"STFT không vuông: Nw={Nw}, F={F}, T={T}"
        )

    return {
        "Nw": Nw,
        "NFFT": NFFT,
        "F": F,
        "H": H,
        "No": No,
        "T": T,
        "overlap_percent": 100.0 * No / Nw,
        "Tw_ms": 1000.0 * Nw / FS,
        "delta_t_ms": 1000.0 * H / FS,
        "delta_f_Hz": FS / NFFT,
    }


CONFIGS = [
    calculate_square_parameters(SEGMENT_LENGTH, Nw)
    for Nw in WINDOW_LENGTHS
]

config_df = pd.DataFrame(CONFIGS)
display(config_df)

In [ ]:
def create_frames(x, Nw, H, T):
    frames = np.zeros((T, Nw), dtype=np.float64)

    for m in range(T):
        start = m * H

        for n in range(Nw):
            frames[m, n] = x[start + n]

    return frames


def apply_window(frames, window):
    T, Nw = frames.shape
    result = np.zeros_like(frames)

    for m in range(T):
        for n in range(Nw):
            result[m, n] = frames[m, n] * window[n]

    return result


def create_dft_basis(Nw, F):
    k_values = np.arange(F, dtype=np.float64).reshape(F, 1)
    n_values = np.arange(Nw, dtype=np.float64).reshape(1, Nw)

    angle = 2.0 * np.pi * k_values * n_values / Nw

    return np.cos(angle), -np.sin(angle)


def manual_dft(windowed_frames, cos_matrix, minus_sin_matrix):
    real_frames = windowed_frames @ cos_matrix.T
    imag_frames = windowed_frames @ minus_sin_matrix.T

    return real_frames.T, imag_frames.T


def calculate_magnitude(real_part, imag_part):
    return np.sqrt(real_part ** 2 + imag_part ** 2)


def magnitude_to_db(magnitude):
    return 20.0 * np.log10(magnitude + 1e-12)

In [ ]:
def zscore_manual(matrix):
    rows, cols = matrix.shape
    count = rows * cols

    total = 0.0

    for r in range(rows):
        for c in range(cols):
            total += float(matrix[r, c])

    mean_value = total / count

    variance_sum = 0.0

    for r in range(rows):
        for c in range(cols):
            d = matrix[r, c] - mean_value
            variance_sum += d * d

    std_value = np.sqrt(variance_sum / count)

    Z = np.zeros_like(matrix)

    for r in range(rows):
        for c in range(cols):
            Z[r, c] = (
                matrix[r, c] - mean_value
            ) / (std_value + 1e-12)

    return Z, mean_value, std_value


def save_stft_png(Z, output_path):
    plt.imsave(
        output_path,
        Z,
        origin="lower",
        cmap="viridis",
    )


def read_mafaulda_csv(csv_path):
    df = pd.read_csv(csv_path, header=None)

    if df.shape[1] != 8:
        raise ValueError(
            f"{csv_path} có {df.shape[1]} cột, yêu cầu 8 cột."
        )

    df.columns = COLUMNS

    signal_series = pd.to_numeric(
        df.iloc[:, VIBRATION_COLUMN_INDEX],
        errors="coerce",
    )

    invalid_count = int(signal_series.isna().sum())

    if invalid_count > 0:
        raise ValueError(
            f"{csv_path} có {invalid_count} giá trị không hợp lệ."
        )

    return signal_series.to_numpy(dtype=np.float64)

In [ ]:
PRECOMPUTED = {}

for config in CONFIGS:
    Nw = int(config["Nw"])
    F = int(config["F"])

    window = create_hamming(Nw)

    cos_matrix, minus_sin_matrix = create_dft_basis(
        Nw,
        F,
    )

    PRECOMPUTED[Nw] = {
        "window": window,
        "cos_matrix": cos_matrix,
        "minus_sin_matrix": minus_sin_matrix,
    }

print("Đã chuẩn bị 4 cấu hình STFT.")

In [ ]:
# ============================================================
# QUÉT CSV
# ============================================================

class_csv_files = {}

for class_key, class_folder in SOURCE_CLASS_DIRS.items():
    if not class_folder.exists():
        raise FileNotFoundError(
            f"Không tìm thấy: {class_folder}"
        )

    csv_files = sorted(
        class_folder.rglob("*.csv")
    )

    if MAX_FILES_PER_CLASS is not None:
        csv_files = csv_files[:MAX_FILES_PER_CLASS]

    if not csv_files:
        raise FileNotFoundError(
            f"Không có CSV trong: {class_folder}"
        )

    class_csv_files[class_key] = csv_files

    print(
        f"{class_key:12s}: "
        f"{len(csv_files)} file CSV"
    )

In [ ]:
# ============================================================
# CÂN BẰNG + SPLIT CỐ ĐỊNH
# ============================================================

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SPLIT_FILE = OUTPUT_ROOT / "data_split.json"
SPLIT_CSV = OUTPUT_ROOT / "data_split.csv"


def speed_value(path):
    try:
        return float(path.stem)
    except ValueError:
        return path.stem


def pick_evenly_spaced(items, count):
    items = list(items)

    if count >= len(items):
        return items

    positions = np.linspace(
        0,
        len(items) - 1,
        count,
        dtype=int,
    )

    return [items[i] for i in positions]


file_split_map = {}
split_rows = []

if SPLIT_FILE.exists():
    print("Đã có data_split.json -> dùng lại split cũ.")

    with open(
        SPLIT_FILE,
        "r",
        encoding="utf-8",
    ) as f:
        split_json = json.load(f)

    restored = {}

    for class_key in SOURCE_CLASS_DIRS:
        restored[class_key] = []

        for split_name in ["train", "validation", "test"]:
            relative_files = (
                split_json["classes"][class_key][split_name]
            )

            for relative_file in relative_files:
                csv_path = (
                    SOURCE_CLASS_DIRS[class_key]
                    / relative_file
                )

                file_split_map[
                    str(csv_path.resolve())
                ] = split_name

                restored[class_key].append(csv_path)

                split_rows.append(
                    {
                        "class": class_key,
                        "file": str(relative_file),
                        "speed_file": csv_path.stem,
                        "split": split_name,
                    }
                )

    class_csv_files = restored

else:
    print("Chưa có data_split.json -> tạo split mới.")

    min_files = min(
        len(v)
        for v in class_csv_files.values()
    )

    n_train = int(min_files * TRAIN_RATIO)
    n_validation = int(min_files * VALIDATION_RATIO)
    n_test = min_files - n_train - n_validation

    split_json = {
        "files_per_class": min_files,
        "number_train": n_train,
        "number_validation": n_validation,
        "number_test": n_test,
        "classes": {},
    }

    balanced = {}

    for class_key, files in class_csv_files.items():
        files = sorted(
            files,
            key=speed_value,
        )

        files = pick_evenly_spaced(
            files,
            min_files,
        )

        test_files = pick_evenly_spaced(
            files,
            n_test,
        )

        test_set = set(test_files)

        remain = [
            p for p in files
            if p not in test_set
        ]

        validation_files = pick_evenly_spaced(
            remain,
            n_validation,
        )

        val_set = set(validation_files)

        train_files = [
            p for p in remain
            if p not in val_set
        ]

        balanced[class_key] = (
            train_files
            + validation_files
            + test_files
        )

        split_json["classes"][class_key] = {}

        for split_name, split_files in [
            ("train", train_files),
            ("validation", validation_files),
            ("test", test_files),
        ]:
            relative_files = []

            for csv_path in split_files:
                relative_file = csv_path.relative_to(
                    SOURCE_CLASS_DIRS[class_key]
                )

                relative_files.append(
                    str(relative_file)
                )

                file_split_map[
                    str(csv_path.resolve())
                ] = split_name

                split_rows.append(
                    {
                        "class": class_key,
                        "file": str(relative_file),
                        "speed_file": csv_path.stem,
                        "split": split_name,
                    }
                )

            split_json["classes"][class_key][
                split_name
            ] = relative_files

        print(
            f"{class_key:12s} | "
            f"Train={len(train_files)} | "
            f"Validation={len(validation_files)} | "
            f"Test={len(test_files)}"
        )

    class_csv_files = balanced

    with open(
        SPLIT_FILE,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            split_json,
            f,
            ensure_ascii=False,
            indent=2,
        )

split_df = pd.DataFrame(split_rows)

split_df.to_csv(
    SPLIT_CSV,
    index=False,
    encoding="utf-8-sig",
)

display(
    split_df.groupby(
        ["split", "class"]
    ).size().rename(
        "number_of_files"
    ).reset_index()
)

In [ ]:
def get_split_name(csv_path):
    key = str(csv_path.resolve())

    if key not in file_split_map:
        raise RuntimeError(
            f"Không tìm thấy split cho: {csv_path}"
        )

    return file_split_map[key]

In [ ]:
# ============================================================
# RESUME THEO SỐ LƯỢNG FILE NPY
# ============================================================

def variant_name_from_config(config):
    return (
        f"Nw_{int(config['Nw'])}_"
        f"H_{int(config['H'])}_"
        f"{int(config['F'])}x{int(config['T'])}"
    )


def check_source_file_complete(
    source_output,
    num_segments=EXPECTED_SEGMENTS,
    verbose=True,
):
    complete = True
    total_npy = 0

    for config in CONFIGS:
        variant_name = variant_name_from_config(
            config
        )

        variant_folder = (
            source_output
            / variant_name
        )

        if not variant_folder.exists():
            count = 0
        else:
            count = len(
                list(
                    variant_folder.glob(
                        "segment_*.npy"
                    )
                )
            )

        total_npy += count

        if verbose:
            print(
                f"    {variant_name:28s}: "
                f"{count}/{num_segments}"
            )

        if count != num_segments:
            complete = False

    if verbose:
        print(
            f"    Tổng NPY: "
            f"{total_npy}/"
            f"{num_segments * len(CONFIGS)}"
        )

    return complete


print(
    "Mỗi file tốc độ phải có:",
    EXPECTED_SEGMENTS,
    "segment ×",
    len(CONFIGS),
    "cửa sổ =",
    EXPECTED_SEGMENTS * len(CONFIGS),
    "NPY"
)

In [ ]:
dataset_summary = []
error_rows = []

total_start = time.perf_counter()

print("Bắt đầu tạo STFT...")

In [ ]:
# ============================================================
# TẠO STFT + RESUME
# ============================================================

for class_key, csv_files in class_csv_files.items():

    class_source_root = SOURCE_CLASS_DIRS[class_key]

    print("\n" + "#" * 90)
    print(f"ĐANG XỬ LÝ LỚP: {class_key}")
    print("#" * 90)

    for file_index, csv_path in enumerate(
        csv_files,
        start=1,
    ):
        file_start = time.perf_counter()

        try:
            split_name = get_split_name(csv_path)
            speed_key = csv_path.stem

            relative_path = csv_path.relative_to(
                class_source_root
            )

            relative_parent = relative_path.parent

            source_output = (
                OUTPUT_ROOT
                / split_name
                / class_key
                / relative_parent
                / speed_key
            )

            source_output.mkdir(
                parents=True,
                exist_ok=True,
            )

            print(
                f"\n[{file_index:03d}/"
                f"{len(csv_files):03d}] "
                f"[{split_name.upper():10s}] "
                f"{class_key} | "
                f"{relative_path}"
            )

            # KIỂM TRA SỐ LƯỢNG TRƯỚC KHI ĐỌC CSV
            if (
                SAVE_NPY
                and SKIP_EXISTING_NPY
                and check_source_file_complete(
                    source_output,
                    EXPECTED_SEGMENTS,
                    verbose=True,
                )
            ):
                print(
                    "    ĐỦ 4 × 61 = 244 NPY "
                    "-> SKIP TOÀN BỘ FILE."
                )
                continue

            # CHỈ FILE CHƯA ĐỦ MỚI ĐỌC CSV
            signal_data = read_mafaulda_csv(
                csv_path
            )

            total_samples = len(signal_data)

            num_segments = (
                total_samples
                // SEGMENT_LENGTH
            )

            remainder_samples = (
                total_samples
                % SEGMENT_LENGTH
            )

            if MAX_SEGMENTS_PER_FILE is not None:
                num_segments = min(
                    num_segments,
                    int(MAX_SEGMENTS_PER_FILE),
                )

            print(
                f"    samples={total_samples} | "
                f"segments={num_segments} | "
                f"dư={remainder_samples}"
            )

            created_count = 0
            skipped_count = 0

            for segment_index in range(
                num_segments
            ):
                start_sample = (
                    segment_index
                    * SEGMENT_LENGTH
                )

                end_sample = (
                    start_sample
                    + SEGMENT_LENGTH
                )

                raw_segment = signal_data[
                    start_sample:end_sample
                ].copy()

                signal_no_dc = None
                dc_value = None

                for config in CONFIGS:
                    Nw = int(config["Nw"])
                    F = int(config["F"])
                    H = int(config["H"])
                    No = int(config["No"])
                    T = int(config["T"])

                    variant_name = (
                        variant_name_from_config(
                            config
                        )
                    )

                    variant_output = (
                        source_output
                        / variant_name
                    )

                    variant_output.mkdir(
                        parents=True,
                        exist_ok=True,
                    )

                    segment_name = (
                        f"segment_"
                        f"{segment_index + 1:03d}"
                    )

                    npy_path = (
                        variant_output
                        / f"{segment_name}.npy"
                    )

                    png_path = (
                        variant_output
                        / f"{segment_name}.png"
                    )

                    # FILE ĐÃ CÓ -> SKIP
                    if (
                        SAVE_NPY
                        and SKIP_EXISTING_NPY
                        and npy_path.exists()
                    ):
                        skipped_count += 1
                        continue

                    # Chỉ tính DC nếu segment có ít nhất 1 cửa sổ thiếu
                    if signal_no_dc is None:
                        signal_no_dc, dc_value = (
                            remove_dc(raw_segment)
                        )

                    frames = create_frames(
                        signal_no_dc,
                        Nw,
                        H,
                        T,
                    )

                    window = (
                        PRECOMPUTED[Nw]["window"]
                    )

                    windowed_frames = apply_window(
                        frames,
                        window,
                    )

                    cos_matrix = (
                        PRECOMPUTED[Nw]["cos_matrix"]
                    )

                    minus_sin_matrix = (
                        PRECOMPUTED[Nw][
                            "minus_sin_matrix"
                        ]
                    )

                    real_part, imag_part = (
                        manual_dft(
                            windowed_frames,
                            cos_matrix,
                            minus_sin_matrix,
                        )
                    )

                    magnitude = calculate_magnitude(
                        real_part,
                        imag_part,
                    )

                    SdB = magnitude_to_db(
                        magnitude
                    )

                    Z, stft_mean, stft_std = (
                        zscore_manual(SdB)
                    )

                    if Z.shape != (F, T):
                        raise RuntimeError(
                            f"Sai shape {Z.shape}; "
                            f"mong đợi {(F, T)}."
                        )

                    if not np.all(np.isfinite(Z)):
                        raise RuntimeError(
                            "STFT chứa NaN hoặc Inf."
                        )

                    if SAVE_NPY:
                        np.save(
                            npy_path,
                            Z.astype(np.float32),
                        )

                    if SAVE_PNG:
                        save_stft_png(
                            Z,
                            png_path,
                        )

                    created_count += 1

                    dataset_summary.append(
                        {
                            "split": split_name,
                            "class": class_key,
                            "speed_file": speed_key,
                            "source_csv": str(relative_path),
                            "variant": variant_name,
                            "Nw": Nw,
                            "H": H,
                            "No": No,
                            "F": F,
                            "T": T,
                            "segment": segment_index + 1,
                            "start_sample": start_sample,
                            "end_sample": end_sample - 1,
                            "dc_value": dc_value,
                            "stft_mean": stft_mean,
                            "stft_std": stft_std,
                            "status": "created",
                            "npy_path": str(
                                npy_path.relative_to(
                                    OUTPUT_ROOT
                                )
                            ),
                        }
                    )

            elapsed = (
                time.perf_counter()
                - file_start
            )

            print(
                f"    created={created_count} | "
                f"skip={skipped_count} | "
                f"{elapsed:.2f} s"
            )

        except Exception as exc:
            print(
                f"\n*** LỖI FILE: {csv_path}"
            )
            print(repr(exc))

            error_rows.append(
                {
                    "class": class_key,
                    "file": str(csv_path),
                    "error": repr(exc),
                }
            )

In [ ]:
# ============================================================
# LƯU SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    dataset_summary
)

summary_path = (
    OUTPUT_ROOT
    / "dataset_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

config_df.to_csv(
    OUTPUT_ROOT
    / "stft_configurations.csv",
    index=False,
    encoding="utf-8-sig",
)

if error_rows:
    pd.DataFrame(
        error_rows
    ).to_csv(
        OUTPUT_ROOT
        / "processing_errors.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Summary:", summary_path)

In [ ]:
# ============================================================
# KIỂM TRA CUỐI CÙNG TỪ CẤU TRÚC THƯ MỤC
# ============================================================

rows = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    for class_key in SOURCE_CLASS_DIRS:
        class_root = (
            OUTPUT_ROOT
            / split_name
            / class_key
        )

        if not class_root.exists():
            continue

        for config in CONFIGS:
            variant_name = (
                variant_name_from_config(
                    config
                )
            )

            for variant_folder in class_root.rglob(
                variant_name
            ):
                count = len(
                    list(
                        variant_folder.glob(
                            "segment_*.npy"
                        )
                    )
                )

                rows.append(
                    {
                        "split": split_name,
                        "class": class_key,
                        "variant": variant_name,
                        "count": count,
                    }
                )

check_df = pd.DataFrame(rows)

if check_df.empty:
    print("Chưa có NPY.")
else:
    display(
        check_df.groupby(
            ["split", "class", "variant"]
        )["count"].sum().reset_index()
    )